Blank notebook with limited support for MTH 229: no `SymPy` or `Plots`
    

In [4]:

using PlotlyLight
    # Plots.jl like interface

"""
    Plot(f::Function, a, b; kwargs...)

Provides an interface like `Plots.plot` for plotting a function `f`.
"""
function PlotlyLight.Plot(f::Function, a::Real, b::Real; kwargs...)
    x,y = unzip(f, a, b)
    cfg = plotly_config(x, y; kwargs...)
    p = isa(cfg, Config) ? Plot(cfg) : Plot(cfg...)
    p
end

function Plot!(p::Plot, x, y; kwargs...)
    cfg = plotly_config(x,y; kwargs...)
    push!(p.data, cfg)
    p
end

"""
    Plot!(p::Plot, f::Function; kwargs...)

Used to add a new tract to an existing plot. Like `Plots.plot!` *but* the current plot must be passed in. Typical use would look like:

```
let
  p = Plot(sin, 0, 2pi)
  Plot!(p, cos)
end
```
"""
function Plot!(p::Plot, f::Function; kwargs...)
    m,M = (Inf, -Inf)
    for d ∈ p.data
        a,b = extrema(d.x)
        m = min(a, m); M = max(b,M)
    end
    x = range(m, M, length=251)
    y = f.(x)
    Plot!(p, x, y; kwargs...)
end

# clean arguments from Plots
function plotly_config(x,y=nothing;
        linewidth=nothing,
                       series = nothing,
                       color = nothing,
                       kwargs...)
    c = Config(;x)
    c.line = Config()
    !isnothing(y) && (c.y = y)

    !isnothing(linewidth) && (c.line.width=linewidth)
    !isnothing(series) && (c.mode = series)
    !isnothing(color) && (c.line.color=color)

    c
end

    ## plotif
    function identify_colors(g, xs, colors=(:red, :blue, :black))
        F = (a,b) -> begin
            ga,gb=g(a),g(b)
            ga * gb < 0 && return nothing
            ga >= 0 && return true
            return false
        end
        find_colors(F, xs, colors)
    end

    # F(a,b) returns true, false, or nothing
    function find_colors(F, xs, colors=(:red, :blue, :black))
        n = length(xs)
        cols = repeat([colors[1]], n)
        for i in 1:n-1
            a,b = xs[i], xs[i+1]
            val = F(a,b)
            if val == nothing
                cols[i] = colors[3]
            elseif val
                cols[i] = colors[1]
            else
                cols[i] = colors[2]
            end
        end
        cols[end] = cols[end-1]
        cols
    end


    function rle(v::Vector{T}) where {T} # stats base
        n = length(v)
        vals = T[]
        lens = Int[]

        n>0 || return (vals,lens)

        cv = v[1]
        cl = 1

        i = 2
        @inbounds while i <= n
            vi = v[i]
            if vi == cv
                cl += 1
            else
                push!(vals, cv)
                push!(lens, cl)
                cv = vi
                cl = 1
            end
            i += 1
        end

        # the last section
        push!(vals, cv)
        push!(lens, cl)

        return (vals, lens)
    end
    function plotif(f,g, a, b; kwargs...)
	xs = collect(range(a, b, length=251))
	cs = identify_colors(g, xs)
	cols, l = rle(cs)
	xs′ = cumsum(l); pushfirst!(xs′, 1)
	p = Plot()
	for i ∈ eachindex(cols)
	    us = xs[xs′[i]:xs′[i+1]]
	    push!(p.data, Config(x=us, y=f.(us), line=Config(color=cols[i])))
	end
	p
    end



plotif (generic function with 1 method)

In [5]:
	# startup commands
	# julia startup package
	# MTH229 alternatives
    using Roots
    using QuadGK
    using SpecialFunctions
    import PlotUtils
	using ForwardDiff # sadly no lighterweight alternative
    Base.adjoint(f::Function) = x -> ForwardDiff.derivative(f, float(x))


    tangent(f, c) = x -> f(c) + f'(c)*(x-c)
    secant(f, a, b) = x -> f(a) + (f(b)-f(a)) / (b-a) * (x - a)
    fisheye(f)=x->atan(f(tan(x)))
    newton(f, x; kwargs...) = find_zero((f, f'), x, Roots.Newton(); kwargs...)
    rangeclamp(f, hi=20, lo=-hi; replacement=NaN) = x -> lo < f(x) < hi ? f(x) : replacement

    ##
function sign_chart(f, a, b; atol=1e-6)
    pm(x) = x < 0 ? "-" : x > 0 ? "+" : "0"
    summarize(f,cp,d) = (DNE_0_∞=cp, sign_change=pm(f(cp-d)) * " → " * pm(f(cp+d)))
    # check endpoint
    if min(abs(f(a)), abs(f(b))) <= max(max(a,b)*eps(), atol)
        return "Sorry, the endpoints must not be zeros for the function"
    end

    zs = find_zeros(f, a, b)
    pts = vcat(a, zs, b)
    for (u,v) ∈ zip(pts[1:end-1], pts[2:end])
        zs′ = find_zeros(x -> 1/f(x), u, v)
        for z′ ∈ zs′
            flag = false
            for z ∈ zs
                if isapprox(z′, z; atol=atol)
                    flag = true
                    break
                end
            end
            !flag && push!(zs, z′)
        end
    end
    if isempty(zs)
	fc = f(a + (b-a)/2)
	return "No sign change, always " * (fc > 0 ? "positive" : iszero(fc) ? "zero" : "negative")
    end

    sort!(zs)
    m,M = extrema(zs)
    d = min((m-a)/2, (b-M)/2)
    if length(zs) > 1
        d′ = minimum(diff(zs))/2
        d = min(d, d′ )
    end
    summarize.([f], zs, d)
end
    function lim(f::Function, c::Real; n::Int=6, dir="+")
	 hs = [(1/10)^i for i in 1:n] # close to 0
	 if dir == "+"
	   xs = c .+ hs
	 else
	   xs = c .- hs
	 end
	 ys = map(f, xs)
	 [xs ys]
end

    ##
function riemann(f::Function, a::Real, b::Real, n::Int; method="right")
  if method == "right"
     meth = (f,l,r) -> f(r) * (r-l)
  elseif method == "left"
     meth= (f,l,r) -> f(l) * (r-l)
  elseif method == "trapezoid"
     meth = (f,l,r) -> (1/2) * (f(l) + f(r)) * (r-l)
  elseif method == "simpsons"
     meth = (f,l,r) -> (1/6) * (f(l) + 4*(f((l+r)/2)) + f(r)) * (r-l)
  end

    xs = range(a, b, length=n+1)
    lrₛ = zip(Iterators.take(xs, n), Iterators.drop(xs, 1))
    sum(meth(f, l, r) for (l,r) in lrₛ)
end
    ##
function bisection(f::Function, a, b)
    a,b = sort([a,b])

    if f(a) * f(b) > 0
        error("[a,b] is not a bracket. A bracket means f(a) and f(b) have different signs!")
    end

    M = a + (b-a) / 2


    i, j = 0, 64
    ss = fill("#", 65)
    ss[i+1]="a"; ss[j+1]="b"
    println("")
    println(join(ss))
    flag = true

    while a < M < b
        if flag && j-i == 1
            ss = fill(" ", 65)
            ss[j:(j+1)] .= "⋮"
            println(join(ss))
            println("")
            flag = false
        end


        if f(M) == 0.0
            println("... exact answer found ...")
	    break
        end
        ## update step
	if f(a) * f(M) < 0
	    a, b = a, M

            if flag
                j = div(i + j, 2)
            end


	else
	    a, b = M, b

            if flag
                i = div(i + j, 2)
            end

	end

        if flag
            ss = fill(".", 65)
            ss[i+1]="a"; ss[j+1]="b"; ss[(i+2):j] .= "#"
            println(join(ss))
        end

        M = a + (b-a) / 2
    end
    M
end
    ##
unzip(vs) = Tuple([vs[j][i] for j in eachindex(vs)] for i in eachindex(vs[1]))
unzip(v,vs...) = unzip([v, vs...])
unzip(r::Function, a, b, n) = unzip(r.(range(a, stop=b, length=n)))
# return (xs, f.(xs)) or (f₁(xs), f₂(xs), ...)
function unzip(f::Function, a, b)
    n = length(f(a))
    if n == 1
        return PlotUtils.adapted_grid(f, (a,b))
    else
        xsys = [PlotUtils.adapted_grid(x->f(x)[i], (a,b)) for i ∈ 1:n]
        xs = sort(vcat([xsys[i][1] for i ∈ 1:n]...))
        return unzip(f.(xs))
    end
end


unzip (generic function with 4 methods)